# 15 — Shopping Assistant Intelligence

## Objective

Transform the existing Top-5 next-basket recommendations into
client-facing shopping intelligence.

This notebook does not train another machine learning model.

It adds:

- Reorder vs new-product discovery classification
- Reorder timing status
- Discovery source
- Client-readable recommendation explanations
- Confidence labels

The resulting table will be used later by the Smart Shopping Assistant application.

In [0]:
from pyspark.sql import functions as F


# Existing final Top-5 recommendations
recommendations_table = (
    "workspace.ml_data.next_basket_serving_recommendations"
)

# Existing next-basket feature table
features_table = (
    "workspace.ml_data.next_basket_features"
)

print("Recommendations table:", recommendations_table)
print("Features table:", features_table)

Recommendations table: workspace.ml_data.next_basket_serving_recommendations
Features table: workspace.ml_data.next_basket_features


In [0]:
recommendations_df = spark.table(
    recommendations_table
)

features_df = spark.table(
    features_table
)


print(
    "Recommendations:",
    recommendations_df.count()
)

print(
    "Feature rows:",
    features_df.count()
)

Recommendations: 131800
Feature rows: 13519765


In [0]:
assistant_features_df = (
    features_df
    .select(
        "user_id",
        "target_order_id",
        "product_id",

        # Candidate information
        "is_reorder_candidate",
        "is_aisle_candidate",
        "is_copurchase_candidate",
        "candidate_source_count",

        # Customer-product history
        "user_product_order_count",
        "user_product_reorder_rate",
        "orders_since_last_product_purchase",
        "user_product_avg_order_gap",
        "user_product_due_score",
        "bought_in_last_order",
        "purchases_last_3_orders",

        

        # Preference / discovery signals
        "customer_aisle_affinity",
        "customer_department_affinity",
        "aisle_candidate_score",
        "copurchase_candidate_score",

        # Whether reorder cycle exists
        "has_reorder_cycle"
    )
    .dropDuplicates(
        [
            "user_id",
            "target_order_id",
            "product_id"
        ]
    )
)

display(
    assistant_features_df.limit(10)
)

user_id,target_order_id,product_id,is_reorder_candidate,is_aisle_candidate,is_copurchase_candidate,candidate_source_count,user_product_order_count,user_product_reorder_rate,orders_since_last_product_purchase,user_product_avg_order_gap,user_product_due_score,bought_in_last_order,purchases_last_3_orders,customer_aisle_affinity,customer_department_affinity,aisle_candidate_score,copurchase_candidate_score,has_reorder_cycle
143901,457087,44359,1,0,0,1,1,0.0,5,0.0,0.0,0,0,0.0635,0.5238,0.0,0.0,0
187881,266719,128,1,0,0,1,2,0.5,3,2.0,1.5,0,1,0.068,0.1068,0.0,0.0,1
155317,1966889,25659,1,0,0,1,1,0.0,16,0.0,0.0,0,0,0.0741,0.1728,0.0,0.0,0
123484,2138293,21288,1,0,0,1,1,0.0,1,0.0,0.0,1,1,0.0571,0.0857,0.0,0.0,0
174428,2407367,44359,1,0,0,1,1,0.0,28,0.0,0.0,0,0,0.1643,0.2657,0.0,0.0,0
182194,3292949,5285,1,0,0,1,2,0.5,9,23.0,0.39,0,0,0.0196,0.1066,0.0,0.0,1
123484,2138293,36425,1,0,0,1,1,0.0,6,0.0,0.0,0,0,0.1429,0.2286,0.0,0.0,0
27821,2782044,35303,1,0,0,1,1,0.0,8,0.0,0.0,0,0,0.0261,0.0957,0.0,0.0,0
145072,1014935,38141,1,0,0,1,1,0.0,36,0.0,0.0,0,0,0.2388,0.2388,0.0,0.0,0
147403,659212,40063,1,0,0,1,1,0.0,36,0.0,0.0,0,0,0.01,0.26,0.0,0.0,0


In [0]:
assistant_df = (
    recommendations_df.alias("r")
    .join(
        assistant_features_df.alias("f"),
        on=[
            "user_id",
            "target_order_id",
            "product_id"
        ],
        how="left"
    )
)

print(
    "Rows after enrichment:",
    assistant_df.count()
)

display(
    assistant_df.limit(10)
)

Rows after enrichment: 131800


user_id,target_order_id,product_id,recommendation_rank,product_name,aisle,department,purchase_probability,is_new_to_customer,candidate_source,recommendation_strategy,is_reorder_candidate,is_aisle_candidate,is_copurchase_candidate,candidate_source_count,user_product_order_count,user_product_reorder_rate,orders_since_last_product_purchase,user_product_avg_order_gap,user_product_due_score,bought_in_last_order,purchases_last_3_orders,customer_aisle_affinity,customer_department_affinity,aisle_candidate_score,copurchase_candidate_score,has_reorder_cycle
21,1854765,28204,3,Organic Fuji Apple,fresh fruits,produce,0.33565992748081386,0,reorder,conditional_hybrid_0.04,1,0,0,1,6,0.8333,1,4.4,0.23,1,2,0.1073,0.1463,0.0,0.0,1
21,1854765,48988,4,Unsweetened Premium Iced Tea,tea,beverages,0.27522307917062994,0,reorder,conditional_hybrid_0.04,1,0,0,1,18,0.9444,3,1.65,1.82,0,1,0.1171,0.2439,0.0,0.0,1
21,1854765,44632,2,Sparkling Water Grapefruit,water seltzer sparkling water,beverages,0.3617915822651947,0,reorder,conditional_hybrid_0.04,1,0,0,1,6,0.8333,2,3.2,0.63,0,2,0.0585,0.2439,0.0,0.0,1
21,1854765,23729,1,Hard Boiled Eggs,eggs,dairy eggs,0.5683535426858906,0,reorder,conditional_hybrid_0.04,1,0,0,1,21,0.9524,2,1.55,1.29,0,2,0.1268,0.2488,0.0,0.0,1
14,2316178,37266,4,Tater Treats Seasoned Shredded Potatoes,frozen appetizers sides,frozen,0.36486852088734056,0,reorder,conditional_hybrid_0.04,1,0,0,1,5,0.8,1,1.5,0.67,1,3,0.0619,0.1667,0.0,0.0,1
14,2316178,29509,1,80 Vodka Holiday Edition,spirits,alcohol,0.6871980336715175,0,reorder,conditional_hybrid_0.04,1,0,0,1,13,0.9231,1,1.0,1.0,1,3,0.0619,0.0619,0.0,0.0,1
21,1854765,33894,5,Goldfish Cheddar Baked Snack Crackers Multi Packs,crackers,snacks,0.24767552364038758,0,reorder,conditional_hybrid_0.04,1,0,0,1,3,0.6667,2,1.5,1.33,0,2,0.0585,0.1951,0.0,0.0,1
14,2316178,15869,5,Sweet Hot Dog Buns,buns rolls,bakery,0.293682771356237,0,reorder,conditional_hybrid_0.04,1,0,0,1,3,0.6667,1,1.0,1.0,1,3,0.0143,0.0667,0.0,0.0,1
14,2316178,23803,2,Jalapeno Pepper,fresh vegetables,produce,0.6460582185396573,0,reorder,conditional_hybrid_0.04,1,0,0,1,12,0.9167,1,1.0,1.0,1,3,0.1095,0.1524,0.0,0.0,1
14,2316178,8744,3,Mixed Vegetables,frozen produce,frozen,0.5080867030927294,0,reorder,conditional_hybrid_0.04,1,0,0,1,8,0.875,1,1.14,0.88,1,3,0.0762,0.1667,0.0,0.0,1


In [0]:
display(
    assistant_df
    .groupBy(
        "is_new_to_customer",
        "candidate_source"
    )
    .count()
    .orderBy(
        "is_new_to_customer",
        F.desc("count")
    )
)

is_new_to_customer,candidate_source,count
0,reorder,129812
1,copurchase,1200
1,aisle+copurchase,671
1,aisle,117


In [0]:
# ============================================================
# 4. Create client-facing shopping intelligence
# ============================================================

assistant_intelligence_df = (
    assistant_df

    # --------------------------------------------------------
    # A. Recommendation type
    # --------------------------------------------------------
    .withColumn(
        "recommendation_type",
        F.when(
            F.col("is_new_to_customer") == 1,
            F.lit("NEW_DISCOVERY")
        ).otherwise(
            F.lit("REORDER")
        )
    )

    # --------------------------------------------------------
    # B. Reorder timing status
    # --------------------------------------------------------
    .withColumn(
        "reorder_status",

        # New product -> no personal reorder history
        F.when(
            F.col("is_new_to_customer") == 1,
            F.lit("NOT_APPLICABLE")
        )

        # Product was in the latest basket
        .when(
            F.col("bought_in_last_order") == 1,
            F.lit("JUST_PURCHASED")
        )

        # No established personal purchase cycle
        .when(
            F.col("has_reorder_cycle") == 0,
            F.lit("NO_ESTABLISHED_CYCLE")
        )

        # Earlier than usual
        .when(
            F.col("user_product_due_score") < 0.75,
            F.lit("EARLY")
        )

        # Getting close to normal reorder time
        .when(
            F.col("user_product_due_score") < 0.95,
            F.lit("DUE_SOON")
        )

        # Around normal reorder time
        .when(
            F.col("user_product_due_score") <= 1.25,
            F.lit("DUE_NOW")
        )

        # Later than normal purchase cycle
        .otherwise(
            F.lit("OVERDUE")
        )
    )
)

print("Shopping intelligence created successfully.")

Shopping intelligence created successfully.


In [0]:
display(
    assistant_intelligence_df
    .select(
        "user_id",
        "recommendation_rank",
        "product_name",
        "purchase_probability",
        "recommendation_type",
        "orders_since_last_product_purchase",
        "user_product_avg_order_gap",
        "user_product_due_score",
        "reorder_status",
        "candidate_source"
    )
    .orderBy(
        "user_id",
        "recommendation_rank"
    )
    .limit(30)
)

user_id,recommendation_rank,product_name,purchase_probability,recommendation_type,orders_since_last_product_purchase,user_product_avg_order_gap,user_product_due_score,reorder_status,candidate_source
14,1,80 Vodka Holiday Edition,0.6871980336715175,REORDER,1,1.0,1.0,JUST_PURCHASED,reorder
14,2,Jalapeno Pepper,0.6460582185396573,REORDER,1,1.0,1.0,JUST_PURCHASED,reorder
14,3,Mixed Vegetables,0.5080867030927294,REORDER,1,1.14,0.88,JUST_PURCHASED,reorder
14,4,Tater Treats Seasoned Shredded Potatoes,0.36486852088734056,REORDER,1,1.5,0.67,JUST_PURCHASED,reorder
14,5,Sweet Hot Dog Buns,0.293682771356237,REORDER,1,1.0,1.0,JUST_PURCHASED,reorder
21,1,Hard Boiled Eggs,0.5683535426858906,REORDER,2,1.55,1.29,OVERDUE,reorder
21,2,Sparkling Water Grapefruit,0.3617915822651947,REORDER,2,3.2,0.63,EARLY,reorder
21,3,Organic Fuji Apple,0.33565992748081386,REORDER,1,4.4,0.23,JUST_PURCHASED,reorder
21,4,Unsweetened Premium Iced Tea,0.27522307917062994,REORDER,3,1.65,1.82,OVERDUE,reorder
21,5,Goldfish Cheddar Baked Snack Crackers Multi Packs,0.24767552364038758,REORDER,2,1.5,1.33,OVERDUE,reorder


In [0]:
# ============================================================
# 4. Create client-facing shopping intelligence
# ============================================================

assistant_intelligence_df = (
    assistant_df

    # --------------------------------------------------------
    # A. Recommendation type
    # --------------------------------------------------------
    .withColumn(
        "recommendation_type",
        F.when(
            F.col("is_new_to_customer") == 1,
            F.lit("NEW_DISCOVERY")
        ).otherwise(
            F.lit("REORDER")
        )
    )

    # --------------------------------------------------------
    # B. Reorder timing status
    # --------------------------------------------------------
    .withColumn(
        "reorder_status",

        # New product: no personal reorder history
        F.when(
            F.col("is_new_to_customer") == 1,
            F.lit("NOT_APPLICABLE")
        )

        # No reliable historical purchase cycle
        .when(
            F.col("has_reorder_cycle") == 0,
            F.lit("NO_ESTABLISHED_CYCLE")
        )

        # Much earlier than usual
        .when(
            F.col("user_product_due_score") < 0.75,
            F.lit("EARLY")
        )

        # Approaching the normal reorder point
        .when(
            F.col("user_product_due_score") < 0.95,
            F.lit("DUE_SOON")
        )

        # Around the customer's normal reorder point
        .when(
            F.col("user_product_due_score") <= 1.25,
            F.lit("DUE_NOW")
        )

        # Later than the normal reorder point
        .otherwise(
            F.lit("OVERDUE")
        )
    )
)

In [0]:
# ============================================================
# 5. Generate client-readable recommendation explanations
# ============================================================

assistant_intelligence_df = (
    assistant_intelligence_df
    .withColumn(
        "why_recommended",

        # ----------------------------------------------------
        # NEW PRODUCT: aisle + co-purchase
        # ----------------------------------------------------
        F.when(
            (F.col("is_new_to_customer") == 1) &
            (F.col("candidate_source") == "aisle+copurchase"),
            F.lit(
                "New product from a preferred aisle that is also "
                "frequently associated with products in your shopping history."
            )
        )

        # ----------------------------------------------------
        # NEW PRODUCT: co-purchase
        # ----------------------------------------------------
        .when(
            (F.col("is_new_to_customer") == 1) &
            (F.col("candidate_source") == "copurchase"),
            F.lit(
                "New product frequently purchased together with "
                "products from your shopping history."
            )
        )

        # ----------------------------------------------------
        # NEW PRODUCT: preferred aisle
        # ----------------------------------------------------
        .when(
            (F.col("is_new_to_customer") == 1) &
            (F.col("candidate_source") == "aisle"),
            F.lit(
                "New product suggested from one of your preferred aisles."
            )
        )

        # ----------------------------------------------------
        # REORDER: overdue
        # ----------------------------------------------------
        .when(
            F.col("reorder_status") == "OVERDUE",
            F.concat(
                F.lit("You usually buy this product every "),
                F.round(F.col("user_product_avg_order_gap"), 1).cast("string"),
                F.lit(" orders, and it has been "),
                F.col("orders_since_last_product_purchase").cast("string"),
                F.lit(" orders since your last purchase.")
            )
        )

        # ----------------------------------------------------
        # REORDER: due now
        # ----------------------------------------------------
        .when(
            F.col("reorder_status") == "DUE_NOW",
            F.concat(
                F.lit(
                    "This product is around your usual reorder time. "
                    "Your typical gap is "
                ),
                F.round(F.col("user_product_avg_order_gap"), 1).cast("string"),
                F.lit(" orders.")
            )
        )

        # ----------------------------------------------------
        # REORDER: due soon
        # ----------------------------------------------------
        .when(
            F.col("reorder_status") == "DUE_SOON",
            F.concat(
                F.lit(
                    "You are approaching your usual reorder time for this product. "
                    "Typical gap: "
                ),
                F.round(F.col("user_product_avg_order_gap"), 1).cast("string"),
                F.lit(" orders.")
            )
        )

        # ----------------------------------------------------
        # REORDER: early
        # ----------------------------------------------------
        .when(
            F.col("reorder_status") == "EARLY",
            F.concat(
                F.lit("Frequently purchased product with a "),
                F.round(
                    F.col("user_product_reorder_rate") * 100,
                    0
                ).cast("int").cast("string"),
                F.lit("% historical reorder rate.")
            )
        )

        # ----------------------------------------------------
        # No established cycle
        # ----------------------------------------------------
        .when(
            F.col("reorder_status") == "NO_ESTABLISHED_CYCLE",
            F.lit(
                "Previously purchased product with insufficient history "
                "to establish a regular reorder cycle."
            )
        )

        .otherwise(
            F.lit("Recommended from your historical shopping behavior.")
        )
    )
)

In [0]:
display(
    assistant_intelligence_df
    .select(
        "user_id",
        "recommendation_rank",
        "product_name",
        F.round(
            F.col("purchase_probability") * 100,
            1
        ).alias("predicted_probability_pct"),
        "recommendation_type",
        "reorder_status",
        "candidate_source",
        "why_recommended"
    )
    .orderBy(
        "user_id",
        "recommendation_rank"
    )
    .limit(30)
)

user_id,recommendation_rank,product_name,predicted_probability_pct,recommendation_type,reorder_status,candidate_source,why_recommended
14,1,80 Vodka Holiday Edition,68.7,REORDER,DUE_NOW,reorder,This product is around your usual reorder time. Your typical gap is 1.0 orders.
14,2,Jalapeno Pepper,64.6,REORDER,DUE_NOW,reorder,This product is around your usual reorder time. Your typical gap is 1.0 orders.
14,3,Mixed Vegetables,50.8,REORDER,DUE_SOON,reorder,You are approaching your usual reorder time for this product. Typical gap: 1.1 orders.
14,4,Tater Treats Seasoned Shredded Potatoes,36.5,REORDER,EARLY,reorder,Frequently purchased product with a 80% historical reorder rate.
14,5,Sweet Hot Dog Buns,29.4,REORDER,DUE_NOW,reorder,This product is around your usual reorder time. Your typical gap is 1.0 orders.
21,1,Hard Boiled Eggs,56.8,REORDER,OVERDUE,reorder,"You usually buy this product every 1.6 orders, and it has been 2 orders since your last purchase."
21,2,Sparkling Water Grapefruit,36.2,REORDER,EARLY,reorder,Frequently purchased product with a 83% historical reorder rate.
21,3,Organic Fuji Apple,33.6,REORDER,EARLY,reorder,Frequently purchased product with a 83% historical reorder rate.
21,4,Unsweetened Premium Iced Tea,27.5,REORDER,OVERDUE,reorder,"You usually buy this product every 1.7 orders, and it has been 3 orders since your last purchase."
21,5,Goldfish Cheddar Baked Snack Crackers Multi Packs,24.8,REORDER,OVERDUE,reorder,"You usually buy this product every 1.5 orders, and it has been 2 orders since your last purchase."


In [0]:
# ============================================================
# 7. Final client-readable explanations
# ============================================================

assistant_intelligence_df = (
    assistant_intelligence_df
    .withColumn(
        "why_recommended",

        # New product discovered from both signals
        F.when(
            (F.col("is_new_to_customer") == 1) &
            (F.col("candidate_source") == "aisle+copurchase"),
            F.lit(
                "New product from a preferred aisle that is also "
                "frequently associated with products you buy."
            )
        )

        # New product from co-purchase behavior
        .when(
            (F.col("is_new_to_customer") == 1) &
            (F.col("candidate_source") == "copurchase"),
            F.lit(
                "New product frequently purchased together with "
                "products from your shopping history."
            )
        )

        # New product from preferred aisle
        .when(
            (F.col("is_new_to_customer") == 1) &
            (F.col("candidate_source") == "aisle"),
            F.lit(
                "New product suggested from one of your preferred aisles."
            )
        )

        # Overdue reorder
        .when(
            F.col("reorder_status") == "OVERDUE",
            F.concat(
                F.lit("You usually buy this product "),
                F.col("reorder_gap_text"),
                F.lit(", and it has been "),
                F.col(
                    "orders_since_last_product_purchase"
                ).cast("string"),
                F.lit(" orders since your last purchase.")
            )
        )

        # Due now
        .when(
            F.col("reorder_status") == "DUE_NOW",
            F.concat(
                F.lit(
                    "This product is around your usual reorder time: "
                ),
                F.col("reorder_gap_text"),
                F.lit(".")
            )
        )

        # Due soon
        .when(
            F.col("reorder_status") == "DUE_SOON",
            F.concat(
                F.lit(
                    "You are approaching your usual reorder time for this product: "
                ),
                F.col("reorder_gap_text"),
                F.lit(".")
            )
        )

        # Still early
        .when(
            F.col("reorder_status") == "EARLY",
            F.concat(
                F.lit(
                    "A frequent favorite, but it is earlier than your usual reorder time. "
                    "Historical reorder rate: "
                ),
                F.round(
                    F.col("user_product_reorder_rate") * 100,
                    0
                ).cast("int").cast("string"),
                F.lit("%.")
            )
        )

        # No cycle
        .when(
            F.col("reorder_status") == "NO_ESTABLISHED_CYCLE",
            F.lit(
                "Previously purchased product with insufficient history "
                "to establish a regular reorder cycle."
            )
        )

        .otherwise(
            F.lit(
                "Recommended from your historical shopping behavior."
            )
        )
    )
)

In [0]:
# ============================================================
# 7. Add client action labels
# ============================================================

assistant_intelligence_df = (
    assistant_intelligence_df
    .withColumn(
        "client_action",

        F.when(
            F.col("shopping_section") == "REORDER_NOW",
            F.lit("Add again")
        )

        .when(
            F.col("shopping_section") == "COMING_UP",
            F.lit("Remind me later")
        )

        .when(
            F.col("shopping_section") == "FAVORITES_FOR_LATER",
            F.lit("Save for later")
        )

        .when(
            F.col("shopping_section") == "DISCOVER",
            F.lit("Try it")
        )

        .otherwise(
            F.lit("View product")
        )
    )
)

In [0]:
display(
    assistant_intelligence_df
    .groupBy(
        "shopping_section"
    )
    .count()
    .orderBy(
        F.desc("count")
    )
)

shopping_section,count
REORDER_NOW,50986
FREQUENT_FAVORITES,50542
COMING_UP,14780
OTHER_RECOMMENDATIONS,13504
DISCOVER,1988


In [0]:
display(
    assistant_intelligence_df
    .groupBy(
        "recommendation_type",
        "reorder_status",
        "shopping_section"
    )
    .count()
    .orderBy(
        "recommendation_type",
        F.desc("count")
    )
)

recommendation_type,reorder_status,shopping_section,count
NEW_DISCOVERY,NOT_APPLICABLE,DISCOVER,1988
REORDER,EARLY,FREQUENT_FAVORITES,50542
REORDER,DUE_NOW,REORDER_NOW,28071
REORDER,OVERDUE,REORDER_NOW,22915
REORDER,DUE_SOON,COMING_UP,14780
REORDER,NO_ESTABLISHED_CYCLE,OTHER_RECOMMENDATIONS,13504


In [0]:
display(
    assistant_intelligence_df
    .select(
        "user_id",
        "recommendation_rank",
        "product_name",
        F.round(
            F.col("purchase_probability") * 100,
            1
        ).alias("predicted_probability_pct"),
        "recommendation_type",
        "reorder_status",
        "shopping_section",
        "client_action",
        "why_recommended"
    )
    .orderBy(
        "user_id",
        "recommendation_rank"
    )
    .limit(40)
)

user_id,recommendation_rank,product_name,predicted_probability_pct,recommendation_type,reorder_status,shopping_section,client_action,why_recommended
14,1,80 Vodka Holiday Edition,68.7,REORDER,DUE_NOW,REORDER_NOW,Add again,This product is around your usual reorder time. Your typical gap is 1.0 orders.
14,2,Jalapeno Pepper,64.6,REORDER,DUE_NOW,REORDER_NOW,Add again,This product is around your usual reorder time. Your typical gap is 1.0 orders.
14,3,Mixed Vegetables,50.8,REORDER,DUE_SOON,COMING_UP,Keep in mind,You are approaching your usual reorder time for this product. Typical gap: 1.1 orders.
14,4,Tater Treats Seasoned Shredded Potatoes,36.5,REORDER,EARLY,FREQUENT_FAVORITES,Buy again,Frequently purchased product with a 80% historical reorder rate.
14,5,Sweet Hot Dog Buns,29.4,REORDER,DUE_NOW,REORDER_NOW,Add again,This product is around your usual reorder time. Your typical gap is 1.0 orders.
21,1,Hard Boiled Eggs,56.8,REORDER,OVERDUE,REORDER_NOW,Add again,"You usually buy this product every 1.6 orders, and it has been 2 orders since your last purchase."
21,2,Sparkling Water Grapefruit,36.2,REORDER,EARLY,FREQUENT_FAVORITES,Buy again,Frequently purchased product with a 83% historical reorder rate.
21,3,Organic Fuji Apple,33.6,REORDER,EARLY,FREQUENT_FAVORITES,Buy again,Frequently purchased product with a 83% historical reorder rate.
21,4,Unsweetened Premium Iced Tea,27.5,REORDER,OVERDUE,REORDER_NOW,Add again,"You usually buy this product every 1.7 orders, and it has been 3 orders since your last purchase."
21,5,Goldfish Cheddar Baked Snack Crackers Multi Packs,24.8,REORDER,OVERDUE,REORDER_NOW,Add again,"You usually buy this product every 1.5 orders, and it has been 2 orders since your last purchase."


In [0]:
assistant_intelligence_df = (
    assistant_intelligence_df
    .withColumn(
        "reorder_gap_text",
        F.when(
            F.round(F.col("user_product_avg_order_gap"), 1) == 1.0,
            F.lit("every order")
        ).otherwise(
            F.concat(
                F.lit("every "),
                F.round(
                    F.col("user_product_avg_order_gap"), 1
                ).cast("string"),
                F.lit(" orders")
            )
        )
    )
)

In [0]:
# ============================================================
# 8. Create final Smart Shopping Assistant serving dataset
# ============================================================

shopping_assistant_serving_df = (
    assistant_intelligence_df
    .select(
        "user_id",
        "target_order_id",
        "product_id",
        "recommendation_rank",
        "product_name",
        "aisle",
        "department",
        "purchase_probability",
        "recommendation_type",
        "reorder_status",
        "shopping_section",
        "client_action",
        "candidate_source",
        "why_recommended",
        "orders_since_last_product_purchase",
        "user_product_avg_order_gap",
        "user_product_due_score",
        "user_product_reorder_rate",
        "customer_aisle_affinity",
        "customer_department_affinity"
    )
)

In [0]:
print("Rows:", shopping_assistant_serving_df.count())

print(
    "Customers:",
    shopping_assistant_serving_df
    .select("user_id")
    .distinct()
    .count()
)

print(
    "Duplicate recommendation rows:",
    shopping_assistant_serving_df
    .groupBy(
        "user_id",
        "target_order_id",
        "product_id"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

Rows: 131800
Customers: 26360
Duplicate recommendation rows: 0


In [0]:
# ============================================================
# 9. Save Smart Shopping Assistant recommendations
# ============================================================

output_table = "workspace.ml_data.shopping_assistant_recommendations"

(
    shopping_assistant_serving_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(output_table)
)

print("Saved:", output_table)

Saved: workspace.ml_data.shopping_assistant_recommendations


In [0]:
saved_df = spark.table(
    "workspace.ml_data.shopping_assistant_recommendations"
)

print("Saved rows:", saved_df.count())

display(
    saved_df
    .select(
        "user_id",
        "recommendation_rank",
        "product_name",
        "reorder_status",
        "shopping_section",
        "client_action",
        "why_recommended"
    )
    .orderBy("user_id", "recommendation_rank")
    .limit(20)
)

Saved rows: 131800


user_id,recommendation_rank,product_name,reorder_status,shopping_section,client_action,why_recommended
14,1,80 Vodka Holiday Edition,DUE_NOW,REORDER_NOW,Add again,This product is around your usual reorder time: every order.
14,2,Jalapeno Pepper,DUE_NOW,REORDER_NOW,Add again,This product is around your usual reorder time: every order.
14,3,Mixed Vegetables,DUE_SOON,COMING_UP,Remind me later,You are approaching your usual reorder time: every 1.1 orders.
14,4,Tater Treats Seasoned Shredded Potatoes,EARLY,FAVORITES_FOR_LATER,Save for later,"A frequent favorite, but it is earlier than your usual reorder time. Historical reorder rate: 80%."
14,5,Sweet Hot Dog Buns,DUE_NOW,REORDER_NOW,Add again,This product is around your usual reorder time: every order.
21,1,Hard Boiled Eggs,OVERDUE,REORDER_NOW,Add again,"You usually buy this product every 1.6 orders, and it has been 2 orders since your last purchase."
21,2,Sparkling Water Grapefruit,EARLY,FAVORITES_FOR_LATER,Save for later,"A frequent favorite, but it is earlier than your usual reorder time. Historical reorder rate: 83%."
21,3,Organic Fuji Apple,EARLY,FAVORITES_FOR_LATER,Save for later,"A frequent favorite, but it is earlier than your usual reorder time. Historical reorder rate: 83%."
21,4,Unsweetened Premium Iced Tea,OVERDUE,REORDER_NOW,Add again,"You usually buy this product every 1.7 orders, and it has been 3 orders since your last purchase."
21,5,Goldfish Cheddar Baked Snack Crackers Multi Packs,OVERDUE,REORDER_NOW,Add again,"You usually buy this product every 1.5 orders, and it has been 2 orders since your last purchase."


In [0]:
# ============================================================
# Final client-readable recommendation explanations
# ============================================================

assistant_intelligence_df = (
    assistant_intelligence_df

    # Natural-language reorder frequency
    .withColumn(
        "reorder_gap_text",
        F.when(
            F.round(F.col("user_product_avg_order_gap"), 1) == 1.0,
            F.lit("every order")
        )
        .otherwise(
            F.concat(
                F.lit("every "),
                F.round(
                    F.col("user_product_avg_order_gap"), 1
                ).cast("string"),
                F.lit(" orders")
            )
        )
    )

    # Recommendation explanation
    .withColumn(
        "why_recommended",

        # New discovery: aisle + co-purchase
        F.when(
            (F.col("is_new_to_customer") == 1) &
            (F.col("candidate_source") == "aisle+copurchase"),
            F.lit(
                "New product from a preferred aisle that is also "
                "frequently associated with products you buy."
            )
        )

        # New discovery: co-purchase
        .when(
            (F.col("is_new_to_customer") == 1) &
            (F.col("candidate_source") == "copurchase"),
            F.lit(
                "New product frequently purchased together with "
                "products from your shopping history."
            )
        )

        # New discovery: aisle
        .when(
            (F.col("is_new_to_customer") == 1) &
            (F.col("candidate_source") == "aisle"),
            F.lit(
                "New product suggested from one of your preferred aisles."
            )
        )

        # Overdue
        .when(
            F.col("reorder_status") == "OVERDUE",
            F.concat(
                F.lit("You usually buy this product "),
                F.col("reorder_gap_text"),
                F.lit(", and it has been "),
                F.col("orders_since_last_product_purchase").cast("string"),
                F.when(
                    F.col("orders_since_last_product_purchase") == 1,
                    F.lit(" order")
                ).otherwise(
                    F.lit(" orders")
                ),
                F.lit(" since your last purchase.")
            )
        )

        # Due now
        .when(
            F.col("reorder_status") == "DUE_NOW",
            F.concat(
                F.lit(
                    "This product is around your usual reorder time: "
                ),
                F.col("reorder_gap_text"),
                F.lit(".")
            )
        )

        # Due soon
        .when(
            F.col("reorder_status") == "DUE_SOON",
            F.concat(
                F.lit(
                    "You are approaching your usual reorder time: "
                ),
                F.col("reorder_gap_text"),
                F.lit(".")
            )
        )

        # Early
        .when(
            F.col("reorder_status") == "EARLY",
            F.concat(
                F.lit(
                    "A frequent favorite, but it is earlier than your usual reorder time. "
                    "Historical reorder rate: "
                ),
                F.round(
                    F.col("user_product_reorder_rate") * 100,
                    0
                ).cast("int").cast("string"),
                F.lit("%.")
            )
        )

        # No established cycle
        .when(
            F.col("reorder_status") == "NO_ESTABLISHED_CYCLE",
            F.lit(
                "Previously purchased product with insufficient history "
                "to establish a regular reorder cycle."
            )
        )

        .otherwise(
            F.lit(
                "Recommended from your historical shopping behavior."
            )
        )
    )
)